# Tick execution timeline

**Tier:** Docs-CI executable (CPU) · **Expected runtime:** under 10 seconds · **Hardware:** CPU

Tick-aware execution selects quotes at or after the decision/arrival time. Long entries use ask, short entries use bid; latency and spread affect the executable price.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from market_data.loading.ticks import TickBook

t0 = pd.Timestamp("2025-01-02 14:30:00", tz="UTC")
quote_times = pd.to_datetime(
    [
        "2025-01-02 14:30:00.000Z",
        "2025-01-02 14:30:00.020Z",
        "2025-01-02 14:30:00.040Z",
        "2025-01-02 14:30:00.060Z",
    ]
)
bid = np.array([100.00, 100.02, 100.01, 99.98])
ask = bid + 0.04
book = TickBook(quote_times.asi8, bid, ask)
decision = t0 + pd.Timedelta(milliseconds=10)
arrival = decision + pd.Timedelta(milliseconds=20)
quote = book.execution_quote_event_at(arrival)
assert quote is not None
fill_time, fill_bid, fill_ask = quote
print(
    {
        "decision": decision,
        "arrival": arrival,
        "first_quote": fill_time,
        "long_buy_ask": fill_ask,
        "short_sell_bid": fill_bid,
        "spread": fill_ask - fill_bid,
    }
)

Figure: bid and ask quotes across four timestamps, with decision time, execution arrival, and selected long-entry quote marked.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.step(quote_times, bid, where="post", label="bid", color="#b84a4a")
ax.step(quote_times, ask, where="post", label="ask", color="#2369a8")
ax.axvline(decision, color="gray", linestyle=":", label="decision")
ax.axvline(arrival, color="#6c4ca5", linestyle="--", label="execution arrival")
ax.scatter([fill_time], [fill_ask], color="#2369a8", zorder=3, label="long entry quote")
ax.set(
    title="Latency moves execution to the first eligible quote",
    ylabel="Price points",
    xlabel="UTC timestamp",
)
ax.legend(ncol=2)
fig.autofmt_xdate()
plt.show()

## Limitations

This visual uses a four-quote synthetic sequence; it demonstrates quote selection, not an empirical slippage distribution. Stop/target behavior depends on the recorded quote path and configured fill model. See [execution and costs](../../../concepts/execution_and_costs), [tick data API](../../../api/domains/data), and [tick fill reconciliation tests](../../../architecture/runtime/environment_step).